In [1]:
!ollama run qwen2.5


>>> Send a message (/? for help)/bye
... 


In [2]:
!pip install ollama
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.5 MB/s

In [3]:
import ollama
llm = "qwen2.5"

In [4]:
import chromadb

## connect to db
db = chromadb.PersistentClient()

## check existing collections
db.list_collections()

## select a collection
collection_name = "chat_history"
collection = db.get_or_create_collection(name=collection_name,
    embedding_function=chromadb.utils.embedding_functions.DefaultEmbeddingFunction())

In [5]:
from datetime import datetime

def save_chat(lst_msg, collection):
    print("--- Saving Chat ---")

    ## extract chat
    chat = ""
    for m in lst_msg:
        chat += f'{m["role"]}: <<{m["content"]}>>' + '\n\n'

    ## get_idx
    idx = str(collection.count() + 1)

    ## generate info
    p = "Describe the following conversation using only 3 keywords separated by a comma (for example: 'finance, volatility, stocks)."
    tags = ollama.generate(model=llm, prompt=p+"\n"+chat)["response"]
    dic_info = {"tags":tags,
                "date": datetime.today().strftime("%Y-%m-%d"),
                "time": datetime.today().strftime("%H-%M")}

    ## write db
    collection.add(documents=[chat], ids=[idx], metadatas=[dic_info])
    print(f"--- Chat num {idx} saved ---","\n")
    print(dic_info,"\n")
    print(chat)
    print("------------------------")

In [6]:
prompt = "You are an intelligent assistant, provide the best possible answer to user's request."
messages = [{"role":"system", "content":prompt}]

while True:
  ## User
  q = input('🙂 >')
  if q == "quit":
    ### save chat before quitting
    save_chat(lst_msg=messages, collection=collection)
    break
  messages.append( {"role":"user", "content":q} )

  ## Model
  agent_res = ollama.chat(model=llm, messages=messages, tools=[])
  res = agent_res["message"]["content"]

  ## Response
  print("👽 >", f"\x1b[1;30m{res}\x1b[0m")
  messages.append( {"role":"assistant", "content":res} )

🙂 >hi
👽 > Hello! How can I assist you today?
🙂 >i must tell you that im highly allergic to chocolate. i could die
👽 > I'm really sorry to hear that you're highly allergic to chocolate. It's important to take this very seriously and avoid any contact with chocolate or products that might contain it. Here are a few steps you can take:

1. **Read Labels Carefully:** Always check food labels for ingredients, especially when eating out.
2. **Inform Restaurant Staff:** When dining at restaurants, inform the staff about your allergy so they can help prevent cross-contamination.
3. **Carry Medication:** Consider carrying an epinephrine auto-injector (like an EpiPen) prescribed by a doctor in case of accidental exposure.
4. **Emergency Plan:** Have a plan for what to do if you accidentally ingest chocolate or come into contact with it.

If you ever feel unwell, seek medical attention immediately. It's crucial to be cautious and proactive about managing your allergy.
🙂 >quit
--- Saving Chat ---


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.76MiB/s]


--- Chat num 1 saved --- 

{'tags': 'allergy, chocolate, emergency', 'date': '2025-07-07', 'time': '23-51'} 

system: <<You are an intelligent assistant, provide the best possible answer to user's request.>>

user: <<hi>>

assistant: <<Hello! How can I assist you today?>>

user: <<i must tell you that im highly allergic to chocolate. i could die>>

assistant: <<I'm really sorry to hear that you're highly allergic to chocolate. It's important to take this very seriously and avoid any contact with chocolate or products that might contain it. Here are a few steps you can take:

1. **Read Labels Carefully:** Always check food labels for ingredients, especially when eating out.
2. **Inform Restaurant Staff:** When dining at restaurants, inform the staff about your allergy so they can help prevent cross-contamination.
3. **Carry Medication:** Consider carrying an epinephrine auto-injector (like an EpiPen) prescribed by a doctor in case of accidental exposure.
4. **Emergency Plan:** Have a pl

In [7]:
def retrieve_chat(query:str) -> str:
    res_db = collection.query(query_texts=[query])["documents"][0][0:10]
    history = ' '.join(res_db).replace("\n", " ")
    return history

tool_retrieve_chat = {'type':'function', 'function':{
    'name': 'retrieve_chat',
    'dsscription': 'When you knowledge is NOT enough to answer the user, you can use this tool to retrieve chats history.',
    'parameters': {'type': 'object',
                   'required': ['query'],
                   'properties': {
                       'query': {'type':'str', 'description':'Input the user question or the topic of the current chat'}
}}}}

## test
retrieve_chat(query="choco")

"system: <<You are an intelligent assistant, provide the best possible answer to user's request.>>  user: <<hi>>  assistant: <<Hello! How can I assist you today?>>  user: <<i must tell you that im highly allergic to chocolate. i could die>>  assistant: <<I'm really sorry to hear that you're highly allergic to chocolate. It's important to take this very seriously and avoid any contact with chocolate or products that might contain it. Here are a few steps you can take:  1. **Read Labels Carefully:** Always check food labels for ingredients, especially when eating out. 2. **Inform Restaurant Staff:** When dining at restaurants, inform the staff about your allergy so they can help prevent cross-contamination. 3. **Carry Medication:** Consider carrying an epinephrine auto-injector (like an EpiPen) prescribed by a doctor in case of accidental exposure. 4. **Emergency Plan:** Have a plan for what to do if you accidentally ingest chocolate or come into contact with it.  If you ever feel unwell

In [8]:
def final_answer(text:str) -> str:
    return text

tool_final_answer = {'type':'function', 'function':{
    'name': 'final_answer',
    'dsscription': 'Returns a natural language response to the user',
    'parameters': {'type': 'object',
                   'required': ['text'],
                   'properties': {'text': {'type':'str', 'description':'natural language response'}}
}}}

In [9]:
dic_tools = {'retrieve_chat':retrieve_chat,
             'final_answer':final_answer}

In [10]:
def use_tool(agent_res:dict, dic_tools:dict) -> dict:
    ## use tool
    if agent_res["message"].tool_calls is not None:
        for tool in agent_res["message"].tool_calls:
            t_name, t_inputs = tool["function"]["name"], tool["function"]["arguments"]
            if f := dic_tools.get(t_name):
                ### calling tool
                print('🔧 >', f"\x1b[1;31m{t_name} -> Inputs: {t_inputs}\x1b[0m")
                ### tool output
                t_output = f(**tool["function"]["arguments"])
                print(t_output)
                ### final res
                res = t_output
            else:
                print('🤬 >', f"\x1b[1;31m{t_name} -> NotFound\x1b[0m")
    ## don't use tool
    else:
        res = agent_res["message"].content
        t_name, t_inputs = '', ''
    return {'res':res, 'tool_used':t_name, 'inputs_used':t_inputs}

In [11]:
def run_agent(llm, messages, available_tools):
    ## use tools until final answer
    tool_used, local_memory = '', ''
    while tool_used != 'final_answer':
        ### use tool
        try:
            agent_res = ollama.chat(model=llm, messages=messages, tools=[v for v in available_tools.values()])
            dic_res = use_tool(agent_res, dic_tools)
            res, tool_used, inputs_used = dic_res["res"], dic_res["tool_used"], dic_res["inputs_used"]
        ### error
        except Exception as e:
            print("⚠️ >", e)
            res = f"I tried to use {tool_used} but didn't work. I will try something else."
            print("👽 >", f"\x1b[1;30m{res}\x1b[0m")
            messages.append( {"role":"assistant", "content":res} )
        ### update memory
        if tool_used not in ['','final_answer']:
            local_memory += f"\n{res}"
            messages.append( {"role":"user", "content":local_memory} )
            available_tools.pop(tool_used)
            if len(available_tools) == 1:
                messages.append( {"role":"user", "content":"now activate the tool final_answer."} )
        ### tools not used
        if tool_used == '':
            break
    return res

In [12]:
prompt = '''
You are an intelligent assistant, provide the best possible answer to user's request.
You must return natual language response.
When interacting with a user, first you must use the tool 'retrieve_chat' to remember previous chats history.
'''
messages = [{"role":"system", "content":prompt}]

while True:
  ## User
  q = input('🙂 >')
  if q == "quit":
    ### save chat before quitting
    save_chat(lst_msg=messages, collection=collection)
    break
  messages.append( {"role":"user", "content":q} )

  ## Model
  available_tools = {"retrieve_chat":tool_retrieve_chat,"final_answer":tool_final_answer}
  res = run_agent(llm, messages, available_tools)

  ## Response
  print("👽 >", f"\x1b[1;30m{res}\x1b[0m")
  messages.append( {"role":"assistant", "content":res} )

🙂 >hi
👽 > Hello! How can I assist you today?
🙂 >i want to eat the famous Vienna sacher torte. Give me the recipe
🔧 > retrieve_chat -> Inputs: {'query': 'recipe for sacher torte'}
system: <<You are an intelligent assistant, provide the best possible answer to user's request.>>  user: <<hi>>  assistant: <<Hello! How can I assist you today?>>  user: <<i must tell you that im highly allergic to chocolate. i could die>>  assistant: <<I'm really sorry to hear that you're highly allergic to chocolate. It's important to take this very seriously and avoid any contact with chocolate or products that might contain it. Here are a few steps you can take:  1. **Read Labels Carefully:** Always check food labels for ingredients, especially when eating out. 2. **Inform Restaurant Staff:** When dining at restaurants, inform the staff about your allergy so they can help prevent cross-contamination. 3. **Carry Medication:** Consider carrying an epinephrine auto-injector (like an EpiPen) prescribed by a do